# Exploratory Data Analysis
**TFG: Evaluación del riesgo de ciberseguridad en PYMES basada en modelos de Machine Learning explicables**

This notebook covers the exploratory analysis of `model_dataset.csv`, the feature-engineered dataset ready for modeling.

Sections:
1. Dataset overview
2. Target variable analysis (class imbalance)
3. Distribution of numeric features
4. Distribution of encoded categorical features
5. Correlation analysis
6. Feature analysis by target class
7. Key findings summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATASET_PATH = 'data/processed/model_dataset.csv'
df = pd.read_csv(DATASET_PATH)

print(f'Rows:     {len(df):,}')
print(f'Columns:  {len(df.columns)}')
print(f'Features: {len(df.columns) - 1}  (+ 1 target)')

## 1. Dataset Overview

In [ ]:
df.describe().T.round(3)

In [ ]:
# Null check — should be zero after feature engineering
nulls = df.isnull().sum()
print('Null values per column:')
print(nulls[nulls > 0] if nulls.any() else 'None')

## 2. Target Variable — Class Imbalance

The target variable `exploited_in_wild` is highly imbalanced: only ~0.5% of CVEs are confirmed exploited according to CISA KEV.
This must be addressed during modeling using `class_weight='balanced'`.

In [ ]:
vc = df['exploited_in_wild'].value_counts()
total = len(df)

print('Target distribution:')
print(f'  Exploited (True):      {vc[True]:>7,}  ({vc[True]/total*100:.2f}%)')
print(f'  Not exploited (False): {vc[False]:>7,}  ({vc[False]/total*100:.2f}%)')
print(f'  Imbalance ratio:       1 : {round(vc[False]/vc[True])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
labels = ['Not exploited\n(99.50%)', 'Exploited\n(0.50%)']
values = [vc[False], vc[True]]
colors = ['#5B8DB8', '#E07B54']
axes[0].bar(labels, values, color=colors, width=0.5)
axes[0].set_title('Target Variable Distribution')
axes[0].set_ylabel('Number of CVEs')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart (zoomed on exploited)
axes[1].pie(values, labels=labels, colors=colors, autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.savefig('notebooks/figures/01_target_distribution.png', bbox_inches='tight')
plt.show()

## 3. Distribution of Numeric Features

In [ ]:
numeric_features = [
    'base_score_final', 'exploitability_score', 'impact_score',
    'num_cpes', 'total_references', 'days_since_published', 'year_published'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    axes[i].hist(df[col], bins=40, color='#5B8DB8', edgecolor='white', linewidth=0.4)
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

axes[-1].set_visible(False)
plt.suptitle('Distribution of Numeric Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('notebooks/figures/02_numeric_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# CVSS base score distribution — key feature for the model
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(df['base_score_final'], bins=50, color='#5B8DB8', edgecolor='white', linewidth=0.4)
ax.axvline(df['base_score_final'].median(), color='#E07B54', linestyle='--', linewidth=1.5, label=f'Median: {df["base_score_final"].median()}')
ax.axvline(df['base_score_final'].mean(),   color='#2E7D32', linestyle='--', linewidth=1.5, label=f'Mean: {df["base_score_final"].mean():.2f}')
ax.set_title('CVSS Base Score Distribution')
ax.set_xlabel('CVSS Base Score')
ax.set_ylabel('Number of CVEs')
ax.legend()

plt.tight_layout()
plt.savefig('notebooks/figures/03_cvss_base_score.png', bbox_inches='tight')
plt.show()

## 4. Distribution of Encoded Categorical Features

In [ ]:
encoded_features = [
    'attack_vector_score', 'attack_complexity_score',
    'privileges_required_score', 'user_interaction_score',
    'confidentiality_score', 'integrity_score', 'availability_score'
]

# Label maps for readability
label_maps = {
    'attack_vector_score':        {0: 'Unknown', 1: 'Physical', 2: 'Local', 3: 'Adjacent', 4: 'Network'},
    'attack_complexity_score':    {0: 'Unknown', 1: 'High', 2: 'Low'},
    'privileges_required_score':  {0: 'Unknown', 1: 'High', 2: 'Low', 3: 'None'},
    'user_interaction_score':     {0: 'Unknown', 1: 'Required', 2: 'None'},
    'confidentiality_score':      {0: 'None', 1: 'Low', 2: 'High'},
    'integrity_score':            {0: 'None', 1: 'Low', 2: 'High'},
    'availability_score':         {0: 'None', 1: 'Low', 2: 'High'},
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(encoded_features):
    vc = df[col].value_counts().sort_index()
    labels = [label_maps[col].get(k, str(k)) for k in vc.index]
    axes[i].bar(labels, vc.values, color='#5B8DB8', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=15)

axes[-1].set_visible(False)
plt.suptitle('Distribution of Encoded CVSS Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('notebooks/figures/04_encoded_features.png', bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
df_corr = df.copy()
df_corr['exploited_in_wild'] = df_corr['exploited_in_wild'].astype(int)

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.4,
    ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('notebooks/figures/05_correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation with target variable specifically
target_corr = df_corr.corr()['exploited_in_wild'].drop('exploited_in_wild').sort_values()

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#E07B54' if v > 0 else '#5B8DB8' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Target (exploited_in_wild)')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('notebooks/figures/06_target_correlation.png', bbox_inches='tight')
plt.show()

print('Top 5 positively correlated with exploitation:')
print(target_corr.tail(5).to_string())
print()
print('Top 5 negatively correlated with exploitation:')
print(target_corr.head(5).to_string())

## 6. Feature Analysis by Target Class

Comparing distributions between exploited and non-exploited CVEs to understand what differentiates them.

In [ ]:
exploited     = df[df['exploited_in_wild'] == True]
not_exploited = df[df['exploited_in_wild'] == False]

print(f'Exploited CVEs:     {len(exploited):,}')
print(f'Not exploited CVEs: {len(not_exploited):,}')
print()

compare_cols = [
    'base_score_final', 'exploitability_score', 'impact_score',
    'total_references', 'days_since_published', 'exploitability_proxy'
]

print(f'{"Feature":<30} {"Not Exploited (mean)":>22} {"Exploited (mean)":>18}')
print('-' * 72)
for col in compare_cols:
    mean_neg = not_exploited[col].mean()
    mean_pos = exploited[col].mean()
    print(f'{col:<30} {mean_neg:>22.3f} {mean_pos:>18.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(compare_cols):
    axes[i].hist(not_exploited[col], bins=40, alpha=0.6, color='#5B8DB8', label='Not exploited', density=True)
    axes[i].hist(exploited[col],     bins=40, alpha=0.7, color='#E07B54', label='Exploited',     density=True)
    axes[i].set_title(col)
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions: Exploited vs Not Exploited', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('notebooks/figures/07_distributions_by_class.png', bbox_inches='tight')
plt.show()

In [ ]:
# Binary flags comparison — percentage of exploited CVEs per flag
binary_cols = [
    'is_network_based', 'is_low_complexity', 'no_privileges_needed',
    'no_user_interaction', 'high_conf_impact', 'high_integ_impact',
    'high_avail_impact', 'is_ransomware_associated', 'has_cwe'
]

exploit_rate = {}
for col in binary_cols:
    rate_1 = df[df[col] == 1]['exploited_in_wild'].mean() * 100
    rate_0 = df[df[col] == 0]['exploited_in_wild'].mean() * 100
    exploit_rate[col] = {'flag=1': rate_1, 'flag=0': rate_0}

rate_df = pd.DataFrame(exploit_rate).T.round(3)
print('Exploitation rate (%) by binary flag value:')
print(rate_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x = range(len(binary_cols))
width = 0.35
ax.bar([i - width/2 for i in x], rate_df['flag=1'], width=width, label='Flag = 1', color='#E07B54')
ax.bar([i + width/2 for i in x], rate_df['flag=0'], width=width, label='Flag = 0', color='#5B8DB8')
ax.set_xticks(list(x))
ax.set_xticklabels(binary_cols, rotation=30, ha='right')
ax.set_ylabel('Exploitation rate (%)')
ax.set_title('Exploitation Rate by Binary Feature Value')
ax.legend()

plt.tight_layout()
plt.savefig('notebooks/figures/08_binary_flags_exploitation.png', bbox_inches='tight')
plt.show()

In [ ]:
# CVEs published per year — evolution over time
year_counts = df.groupby(['year_published', 'exploited_in_wild']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
year_counts[False].plot(kind='bar', ax=ax, color='#5B8DB8', label='Not exploited')
year_counts[True].plot(kind='bar', ax=ax, color='#E07B54', label='Exploited', alpha=0.85)
ax.set_title('CVEs Published per Year by Exploitation Status')
ax.set_xlabel('Year')
ax.set_ylabel('Number of CVEs')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('notebooks/figures/09_cves_per_year.png', bbox_inches='tight')
plt.show()

## 7. Key Findings Summary

These findings will be discussed in the thesis memory under the Exploratory Analysis section.

In [ ]:
print('KEY FINDINGS')
print('=' * 60)

print(f'\n1. CLASS IMBALANCE')
print(f'   Ratio: 1 exploited for every {round(vc[False]/vc[True])} not exploited CVEs.')
print(f'   Action required: class_weight=balanced in both models.')

print(f'\n2. CVSS BASE SCORE')
print(f'   Mean (not exploited): {not_exploited["base_score_final"].mean():.2f}')
print(f'   Mean (exploited):     {exploited["base_score_final"].mean():.2f}')
print(f'   Exploited CVEs have higher CVSS scores on average.')

print(f'\n3. RANSOMWARE ASSOCIATION')
ransomware_rate = df[df['is_ransomware_associated'] == 1]['exploited_in_wild'].mean() * 100
print(f'   Exploitation rate when ransomware-associated: {ransomware_rate:.1f}%')
print(f'   This is the strongest binary predictor in the dataset.')

print(f'\n4. NETWORK-BASED VULNERABILITIES')
net_rate = df[df['is_network_based'] == 1]['exploited_in_wild'].mean() * 100
print(f'   Exploitation rate for network-based CVEs: {net_rate:.2f}%')

print(f'\n5. TEMPORAL PATTERN')
print(f'   Mean days since published (not exploited): {not_exploited["days_since_published"].mean():.0f}')
print(f'   Mean days since published (exploited):     {exploited["days_since_published"].mean():.0f}')
print(f'   Exploited CVEs tend to be older on average.')